# Facility registry — the dimension table, and what it cannot tell you

`facilities.csv` is the odd one out in this series. The other four notebooks read time series with
hundreds of thousands of rows; this is 176 rows, one snapshot, no time axis at all. Its job is not to
be analysed on its own but to **name and describe the facilities the other files refer to**, so the
useful question is not "what is the distribution of capacity" but "what can this table be trusted to
say, and what does every other notebook get wrong if it trusts it further than that".

Five findings, and four of them are cautions:

- **`System Size (MW)` is a generator-only field.** It is null on exactly the 100 rows that are not
  generators and populated on exactly the 76 that are — a perfect correspondence, so the nulls are a
  definition rather than missing data. The near-empty second capacity column behaves the same way.
- **The registry is a snapshot, and it silently disagrees with the history.** `MUJA_G6` carries 454
  days of temperature readings ending 31 March 2025 and does not appear in the registry at all. An
  inner join from any 2023–2026 series to this file drops retired plant without saying so.
- **There is no fuel or technology column**, so technology has to be inferred from the facility code.
  The `infer_tech` helper in `src/wa_data.py` left **15 generators and 917 MW — 10.4% of registered
  capacity — unclassified**, including the two largest combined-cycle gas plants in the state. Three
  more suffix rules cut that to 11 facilities and 211 MW.
- **Storage capacity is on a different measurement basis from everything else in the column.** For
  six of the seven batteries, `System Size (MW)` equals measured *charge power plus discharge power*
  to within 1%. Read as a generation mix, the registry overstates storage by a factor of two.
- **Ownership is concentrated, and the facility count actively hides it.** 38 participants hold
  capacity, Synergy holds 44.5% of it, and two participants hold half. The largest participant by
  *facility count* — Enel X, with 87 of the 90 Demand Side Programme rows — holds none of it.

In [1]:
# Load the registry, and the two series that can be joined to it
import sys
sys.path.insert(0, '../src')  # Add src directory to path (go up one level from notebooks/)
from wa_data import (load_facilities, load_temperature, load_facility_scada,
                     TECH_SUFFIX, infer_tech)

import glob
import re

import numpy as np
import pandas as pd

f = load_facilities(raw='../data/raw')

# The one column load_facilities does not rename, because it applies to a single
# class of row — established two cells below.
f = f.rename(columns={'Remaining capacity from embedded generator (MW)': 'remaining_mw'})

# The three GENERATING classes. Everything else is a load, a network element or a
# demand-response registration, and this split is the most important fact about
# the table: it is what makes the capacity column readable.
GEN_CLASSES = ['Scheduled Facility', 'Semi-Scheduled Facility', 'Non-Scheduled Facility']
f['is_gen'] = f.facility_class.isin(GEN_CLASSES)
gen = f[f.is_gen].copy()
TOTAL_MW = gen.system_size_mw.sum()

print(f.head().to_string())
print(f'\nshape {f.shape}')
print(f'columns {f.columns.tolist()}')

  participant_code            participant_name  facility_code         facility_class  system_size_mw  remaining_mw     tech  is_gen
0            ALCOA  Alcoa of Australia Limited    ALCOA_KW_IL  Non-Dispatchable Load             NaN          66.0  unknown   False
1            ALCOA  Alcoa of Australia Limited   ALCOA_PNJ_IL  Non-Dispatchable Load             NaN           NaN  unknown   False
2            ALCOA  Alcoa of Australia Limited      ALCOA_WGP     Scheduled Facility            16.0           NaN  unknown    True
3            ALCOA  Alcoa of Australia Limited   ALCOA_WGP_IL  Non-Dispatchable Load             NaN          85.0  unknown   False
4           ALINTA        Alinta Sales Pty Ltd  ALINTA_PNJ_U1     Scheduled Facility           143.0           NaN  thermal    True

shape (176, 8)
columns ['participant_code', 'participant_name', 'facility_code', 'facility_class', 'system_size_mw', 'remaining_mw', 'tech', 'is_gen']


In [2]:
# Key integrity. This is a dimension table: if the keys do not hold, every join in
# every other notebook is unsound. All three hold.
print(f'rows {len(f)}   unique facility codes {f.facility_code.nunique()}   '
      f'duplicated {f.facility_code.duplicated().sum()}')
print(f'participants {f.participant_code.nunique()}   '
      f'code -> name 1:1 {(f.groupby("participant_code").participant_name.nunique() > 1).sum() == 0}   '
      f'name -> code 1:1 {(f.groupby("participant_name").participant_code.nunique() > 1).sum() == 0}')
print(f'facility classes {f.facility_class.nunique()}   '
      f'generators {f.is_gen.sum()}   non-generators {(~f.is_gen).sum()}')
print(f'total registered capacity {TOTAL_MW:,.1f} MW')

print('\nrows and capacity by class:')
by_class = f.groupby('facility_class').agg(
    rows=('facility_code', 'size'),
    with_mw=('system_size_mw', 'count'),
    total_mw=('system_size_mw', 'sum'),
    median_mw=('system_size_mw', 'median'),
    participants=('participant_code', 'nunique')).sort_values('total_mw', ascending=False)
print(by_class.round(1).to_string())

rows 176   unique facility codes 176   duplicated 0
participants 43   code -> name 1:1 True   name -> code 1:1 True
facility classes 7   generators 76   non-generators 100
total registered capacity 8,805.3 MW

rows and capacity by class:
                         rows  with_mw  total_mw  median_mw  participants
facility_class                                                           
Scheduled Facility         47       47    7324.9      118.2            18
Semi-Scheduled Facility    13       13    1418.3       89.1            11
Non-Scheduled Facility     16       16      62.0        2.8            11
Network                     2        0       0.0        NaN             1
Interruptible Load          1        0       0.0        NaN             1
Demand Side Programme      90        0       0.0        NaN             4
Non-Dispatchable Load       7        0       0.0        NaN             5


## 1. The nulls are definitions

`System Size (MW)` is missing on 100 of 176 rows, which looks like the headline data-quality problem
and is not a problem at all. The nulls fall on exactly the four non-generating classes and nowhere
else: **`System Size` is present if and only if the row is a generator**, on all 176 rows. A Demand
Side Programme registration is an obligation to *reduce* load, a Network row is a network element,
and neither has a generating capacity to state. Imputing this column — with a zero, a mean, anything
— would be inventing data, and dropping the null rows would silently discard every load and every
demand-response registration in the market.

The second capacity column makes the same point more sharply. `Remaining capacity from embedded
generator (MW)` is populated on 5 rows of 176, which reads like a broken column. It is not: all five
are **Non-Dispatchable Loads**, 5 of that class's 7 rows, and it is populated nowhere else. It is a
field about industrial sites carrying their own on-site generation — Alcoa's two refineries,
Worsley, Southern Cross — and it is empty everywhere else because everywhere else it does not apply.

The working rule for the rest of the series: **filter to `is_gen` before touching capacity**, and
treat `remaining_mw` as an attribute of Non-Dispatchable Loads rather than a general one.

In [3]:
# STRUCTURAL NULLITY, not missing data.
print(pd.crosstab(f.facility_class, f.system_size_mw.notna(),
                  rownames=['facility class'], colnames=['has System Size']).to_string())
print(f'\n"System Size present" == "is a generator", on every one of {len(f)} rows: '
      f'{(f.system_size_mw.notna() == f.is_gen).all()}')

print('\n`remaining_mw` — every non-null row:')
print(f.loc[f.remaining_mw.notna(),
            ['facility_code', 'facility_class', 'remaining_mw']].to_string(index=False))
ndl = f.facility_class == 'Non-Dispatchable Load'
print(f'\nnon-null inside Non-Dispatchable Load: {f.loc[ndl, "remaining_mw"].notna().sum()} of {ndl.sum()}')
print(f'non-null outside it:                    {f.loc[~ndl, "remaining_mw"].notna().sum()}')

has System Size          False  True 
facility class                       
Demand Side Programme       90      0
Interruptible Load           1      0
Network                      2      0
Non-Dispatchable Load        7      0
Non-Scheduled Facility       0     16
Scheduled Facility           0     47
Semi-Scheduled Facility      0     13

"System Size present" == "is a generator", on every one of 176 rows: True

`remaining_mw` — every non-null row:
    facility_code        facility_class  remaining_mw
      ALCOA_KW_IL Non-Dispatchable Load          66.0
     ALCOA_WGP_IL Non-Dispatchable Load          85.0
NWNMTMN_FIM_INML1 Non-Dispatchable Load           0.2
 WAPL_WORSLEY_IL1 Non-Dispatchable Load          45.0
      STHRNCRS_IL Non-Dispatchable Load          72.0

non-null inside Non-Dispatchable Load: 5 of 7
non-null outside it:                    0


## 2. A snapshot, not a history

Nothing in the file carries a usable date. `Extracted At` is populated on the first row only — quirk
5 in `src/wa_data.py`, a file-level watermark the loader drops — so the registry describes the market
*as it stood when it was fetched* and says nothing about the three years the other notebooks cover.

That is not a theoretical concern, and the temperature file proves it. `MUJA_G6` carries **454 daily
temperature readings from 2 January 2024 to 31 March 2025**, and then stops. It is not in the
registry; only `MUJA_G7` and `MUJA_G8` are. A coal unit retired mid-record and the registry keeps no
trace of it, so an inner join from a historical series to this file drops `MUJA_G6` silently, and an
analyst who never checked the anti-join would conclude those readings had never existed.

The same cell surfaces a smaller version of the problem pointing the other way. `ALBANY_WF1` and
`GRASMERE_WF1` appear in the raw temperature files with a row for every trading date and **not one
non-null reading between them**. They are registered, they are in the panel, and they carry no data.
`load_temperature` drops null readings, so they disappear before reaching any analysis — the right
default, and worth knowing rather than discovering.

In [4]:
# The registry is current; the series are historical. The gap is measurable.
temp_raw = pd.concat([pd.read_csv(p) for p in
                      sorted(glob.glob('../data/raw/facility-temperature-*.csv'))])
temp_raw['Trading Date'] = pd.to_datetime(temp_raw['Trading Date'])
temp = load_temperature(raw='../data/raw')

reg_codes = set(f.facility_code)
raw_codes = set(temp_raw['Facility Code'].unique())
live_codes = set(temp.facility_code.unique())   # after null readings are dropped

print(f'temperature codes in the raw files {len(raw_codes)}   '
      f'with >=1 reading {len(live_codes)}   also in the registry {len(live_codes & reg_codes)}')

orphans = sorted(live_codes - reg_codes)
print(f'\nIN THE HISTORY, NOT IN THE REGISTRY: {orphans}')
for c in orphans:
    g_ = temp_raw[temp_raw['Facility Code'] == c].dropna(subset=['Maximum Daily Temperature'])
    print(f'  {c}: {len(g_)} readings, '
          f'{g_["Trading Date"].min():%Y-%m-%d} to {g_["Trading Date"].max():%Y-%m-%d}')
print(f'  sibling units that ARE registered: {sorted(c for c in reg_codes if "MUJA" in c)}')

empty = sorted(raw_codes - live_codes)
print(f'\nREGISTERED IN THE PANEL, NEVER A READING: {empty}')
print(temp_raw[temp_raw['Facility Code'].isin(empty)].groupby('Facility Code').agg(
    rows=('Maximum Daily Temperature', 'size'),
    readings=('Maximum Daily Temperature', 'count')).to_string())

temperature codes in the raw files 61   with >=1 reading 59   also in the registry 58

IN THE HISTORY, NOT IN THE REGISTRY: ['MUJA_G6']
  MUJA_G6: 454 readings, 2024-01-02 to 2025-03-31
  sibling units that ARE registered: ['MUJA_G7', 'MUJA_G8']

REGISTERED IN THE PANEL, NEVER A READING: ['ALBANY_WF1', 'GRASMERE_WF1']
               rows  readings
Facility Code                
ALBANY_WF1      973         0
GRASMERE_WF1    973         0


## 3. No fuel column, so technology is an assumption

The registry names a participant, a class and a capacity. It does not say what a facility *is* — no
fuel, no technology, no prime mover. `src/wa_data.py` therefore infers technology from the facility
code suffix and labels it a STATED ASSUMPTION, which is the right posture. This cell audits how far
the assumption actually reaches, because an inferred column that quietly fails on the biggest plants
is worse than no column at all.

The naming convention is only **partly** systematic. It is perfectly systematic in one direction:
all 87 `_R` codes are Demand Side Programme rows and every `_R` row is one. But among generators the
shipped patterns — `_BESS`/`_ESR`, `_WF`, `_PV`/`_SF`, `_GT`/`_CCGT`/`_U`/`_G` — leave **15
facilities and 917.1 MW, 10.4% of all registered capacity, as `unknown`**, and the two largest are
`NEWGEN_KWINANA_CCG1` (334.8 MW) and `COCKBURN_CCG1` (240.0 MW). Both are combined-cycle gas plants
that the `_CCGT` pattern misses because the actual suffix in this file is `_CCG`.

Three additions are unambiguous from the code alone — `_CCG` and `_COG` are combined-cycle and
cogeneration, both thermal; `_WWF` is a wind farm — and they cut the residual to **11 facilities and
211.1 MW, 2.4%**, with nothing above 82 MW left unresolved. The remaining eleven are genuinely not
inferable from their codes: `PRK_AG`, `STHRNCRS_EG`, `TAMALA_PARK`, `ROCKINGHAM`, `BIOGAS01` and the
rest would need an external source, so they stay `unknown` rather than being guessed at.

In [5]:
# How far does the inferred `tech` column actually reach?
def mix(frame, col):
    t = frame.groupby(col).system_size_mw.agg(n='count', MW='sum')
    t['MW%'] = 100 * t.MW / TOTAL_MW
    return t.sort_values('MW', ascending=False).round(1)


print('shipped infer_tech, generators only:')
print(mix(gen, 'tech').to_string())
print(f"\nunresolved: {(gen.tech == 'unknown').sum()} facilities, "
      f"{gen.loc[gen.tech == 'unknown', 'system_size_mw'].sum():,.1f} MW")
print(gen[gen.tech == 'unknown'].nlargest(4, 'system_size_mw')[
    ['facility_code', 'participant_name', 'system_size_mw']].to_string(index=False))

# Three unambiguous suffixes the shipped patterns miss. _CCGT never occurs; the
# file spells combined cycle _CCG.
EXTRA_SUFFIX = [(r'_(CCG|COG)\d*$', 'thermal'), (r'_WWF\d*$', 'wind')]


def infer_tech2(code):
    for pattern, tech in list(TECH_SUFFIX) + EXTRA_SUFFIX:
        if re.search(pattern, str(code)):
            return tech
    return 'unknown'


gen['tech2'] = gen.facility_code.map(infer_tech2)
f['tech2'] = f.facility_code.map(infer_tech2)
print('\nwith _CCG, _COG and _WWF added:')
print(mix(gen, 'tech2').to_string())

left = gen[gen.tech2 == 'unknown']
print(f'\nstill unresolved: {len(left)} facilities, {left.system_size_mw.sum():,.1f} MW '
      f'({100 * left.system_size_mw.sum() / TOTAL_MW:.1f}%), largest {left.system_size_mw.max():.0f} MW')
print(left[['facility_code', 'participant_name', 'facility_class', 'system_size_mw']]
      .sort_values('system_size_mw', ascending=False).to_string(index=False))

# The convention IS perfectly systematic for demand response, in both directions.
f['suffix'] = f.facility_code.str.extract(r'_([A-Z]+)\d*$')
print(f"\n_R codes: {(f.suffix == 'R').sum()}, all Demand Side Programme: "
      f"{(f.loc[f.suffix == 'R', 'facility_class'] == 'Demand Side Programme').all()}")
print(f"DSP rows not using _R: "
      f"{f.loc[(f.facility_class == 'Demand Side Programme') & (f.suffix != 'R'), 'facility_code'].tolist()}")

shipped infer_tech, generators only:
          n      MW   MW%
tech                     
thermal  33  3669.0  41.7
storage   7  2850.0  32.4
wind     16  1118.4  12.7
unknown  15   917.1  10.4
solar     5   250.8   2.8

unresolved: 15 facilities, 917.1 MW
      facility_code             participant_name  system_size_mw
NEWGEN_KWINANA_CCG1 NewGen Power Kwinana Pty Ltd           334.8
      COCKBURN_CCG1                      Synergy           240.0
         ALINTA_WWF  Walkaway Wind Power Pty Ltd            89.1
    NAMKKN_MERR_SG1              Merredin Energy            82.0

with _CCG, _COG and _WWF added:
          n      MW   MW%
tech2                    
thermal  36  4285.9  48.7
storage   7  2850.0  32.4
wind     17  1207.5  13.7
solar     5   250.8   2.8
unknown  11   211.1   2.4

still unresolved: 11 facilities, 211.1 MW (2.4%), largest 82 MW
          facility_code               participant_name         facility_class  system_size_mw
        NAMKKN_MERR_SG1                Merred

## 4. Storage is measured differently from everything else in the same column

The storage rows sum to 2,850 MW — 32.4% of all registered capacity, which would make batteries the
second-largest technology in the SWIS. That number is not comparable to the thermal or wind numbers
beside it, and the SCADA record shows why.

Joining each battery's registered `System Size` to its measured dispatch extremes gives a ratio between `System Size` and the *two-sided span* of
**1.000 to 1.010** — maximum discharge minus
maximum charge — for six of the seven units. In other words `System Size (MW)` for a battery is
charge power **plus** discharge power, while for a gas turbine it is simply output. The seventh unit,
`ALINTA_WGP_ESR1`, sits at 0.3 because it only began dispatching in July 2026 and has not yet reached
its rating; the storage notebook establishes that separately.

Put on the same basis as a generator — discharge power only — the fleet is **1,425 MW**, not 2,850.
The corrected mix is the one to quote: storage falls from 32.4% to 19.3% of capacity and thermal
rises from 48.7% to 58.1%. This is the single most consequential thing in the file, because it is
invisible: the column has one name, one unit, and two meanings.

In [6]:
# The registered System Size of a battery is charge power PLUS discharge power.
scada = load_facility_scada(raw='../data/raw')
span = scada.groupby('facility_code').mw.agg(max_discharge='max', max_charge='min')
span['two_sided_span'] = span.max_discharge - span.max_charge
span = span.join(gen.set_index('facility_code').system_size_mw)
span['span / System Size'] = span.two_sided_span / span.system_size_mw
print(span.round(3).to_string())
print('\nratio for the six units past commissioning: '
      f"{span.drop('ALINTA_WGP_ESR1')['span / System Size'].min():.3f} to "
      f"{span.drop('ALINTA_WGP_ESR1')['span / System Size'].max():.3f}")

# Restate the mix on a discharge-power basis, which is what a generator rating means.
STORAGE_MW_REG = gen.loc[gen.tech2 == 'storage', 'system_size_mw'].sum()
adj = gen.copy()
adj.loc[adj.tech2 == 'storage', 'system_size_mw'] /= 2.0
ADJ_TOTAL = adj.system_size_mw.sum()

cmp_ = pd.DataFrame({
    'as registered MW': gen.groupby('tech2').system_size_mw.sum(),
    'comparable MW': adj.groupby('tech2').system_size_mw.sum()})
cmp_['as registered %'] = 100 * cmp_['as registered MW'] / TOTAL_MW
cmp_['comparable %'] = 100 * cmp_['comparable MW'] / ADJ_TOTAL
print(f'\nregistered total {TOTAL_MW:,.1f} MW  ->  comparable total {ADJ_TOTAL:,.1f} MW')
print(cmp_.sort_values('comparable MW', ascending=False).round(1).to_string())

                 max_discharge  max_charge  two_sided_span  system_size_mw  span / System Size
facility_code                                                                                 
ALINTA_WGP_ESR1         25.248     -25.416          50.664           200.0               0.253
COLLIE_BESS2           300.060    -299.964         600.024           600.0               1.000
COLLIE_ESR1            199.980    -200.076         400.056           400.0               1.000
COLLIE_ESR4            252.000    -252.300         504.300           500.0               1.009
COLLIE_ESR5            252.696    -252.456         505.152           500.0               1.010
KWINANA_ESR1           100.668    -100.140         200.808           200.0               1.004
KWINANA_ESR2           226.356    -223.656         450.012           450.0               1.000

ratio for the six units past commissioning: 1.000 to 1.010

registered total 8,805.3 MW  ->  comparable total 7,380.3 MW
         as registered 

## Visualisation

Four views. The registry has no time axis, so none of these is a time series — they are all
questions about composition, spread and coverage.

1. **What the registry counts against what it rates** — the 90-row class that carries no capacity.
2. **Capacity by technology** — every facility as one mark on a log scale spanning three orders of
   magnitude, with the unresolved technologies marked.
3. **Who owns it** — the concentration curve, and the ten largest holders.
4. **What it can be joined to** — coverage of each linked series by facility count and by capacity,
   which are very different numbers.

Identity is carried by **position** throughout — one row per class, per technology, per participant —
so colour is free to carry one meaning and only one: **blue is registered, known, or covered; orange
is absent, unresolved, or uncovered**. That is a two-colour scheme and it holds across all four
charts. The pair is the blue and orange of the demand, DPV, price and storage notebooks, so this
notebook reads as part of the same system; validated on this surface at all-pairs CVD ΔE 24.7
(protan), normal-vision ΔE 33.6, both well clear of the thresholds, and both above 3:1 against the
chart surface.

In [7]:
# ── Chart setup: palette, shared theme ───────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chart chrome, light surface — identical to the demand, DPV, price and storage
# notebooks, so all five read as one system.
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")

# PRESENCE is the only semantic axis in this notebook, so it takes exactly two
# hues and keeps the meaning they carry elsewhere in the series.
BLUE, ORANGE = "#2a78d6", "#eb6834"
KNOWN, ABSENT = BLUE, ORANGE


def style(fig, title, subtitle=None, height=420, hover="closest",
          top=112, bottom=58, left=74, legend_y=1.0, showlegend=True):
    """Shared theme: light surface, recessive grid, muted axes, ink-coloured text."""
    head = f"<b>{title}</b>"
    if subtitle:
        head += f"<br><span style='font-size:12.5px;color:{INK_2}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=17, color=INK), x=0, xanchor="left", y=0.97),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, hovermode=hover,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=12, color=INK_2),
        height=height, margin=dict(t=top, r=30, b=bottom, l=left), showlegend=showlegend,
        legend=dict(orientation="h", yanchor="bottom", y=legend_y, xanchor="left", x=0,
                    bgcolor="rgba(0,0,0,0)", font=dict(size=11.5)))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS,
                     ticks="outside", tickcolor=AXIS, tickfont=dict(color=MUTED))
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS,
                     ticks="outside", tickcolor=AXIS, tickfont=dict(color=MUTED))
    for a in (fig.layout.annotations or []):
        if a.font is None or a.font.size is None:
            a.font = dict(size=12.5, color=INK_2)
    return fig


def shorten(name, n=26):
    """Participant legal names run to 47 characters; axis labels do not."""
    name = re.sub(r'\s+(Pty Ltd|Ltd|Limited|Partnership)\b\.?', '', name).strip()
    name = re.sub(r'\s+A[TF]F\s+.*$', '', name).strip()   # "X ATF The X Trust" -> "X"
    return name if len(name) <= n else name[:n - 1] + '…'


print(f'surface {SURFACE}   known {KNOWN}   absent {ABSENT}')

surface #fcfcfb   known #2a78d6   absent #eb6834


### 1. What the registry counts, against what it rates

Two panels, one row per facility class, sorted so the generating classes come first. The left panel
is how many rows each class has; the right is how much capacity those rows declare. They are drawn as
separate panels on separate scales rather than as one chart with two axes, because a count and a
megawatt figure have no common scale and overlaying them would let the choice of axis manufacture
whatever relationship the reader was looking for.

The disconnect is the whole point. **Demand Side Programme is the largest class by a wide margin — 90
of 176 rows, more than half the file — and it contributes zero registered megawatts**, because
capacity is not a field that applies to it. Scheduled Facility is barely half its size at 47 rows and
carries 7,325 MW, 83% of everything in the registry. The four orange classes, 100 rows between them,
account for the entire "missing" capacity column.

The other thing the left panel is not showing, and the prose must: those 90 Demand Side Programme
rows belong to just **four participants, and 87 of them to one**, Enel X. Counting rows in this file
counts registrations, not market participants and not physical plant.

In [8]:
# ── Chart 1: rows per class, and capacity per class ──────────────────────────
order = (list(by_class[by_class.index.isin(GEN_CLASSES)].sort_values('total_mw', ascending=False).index)
         + list(by_class[~by_class.index.isin(GEN_CLASSES)].sort_values('rows', ascending=False).index))
b = by_class.loc[order]
is_g = b.index.isin(GEN_CLASSES)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.055,
                    subplot_titles=["Rows in the registry", "Registered capacity, MW"])

for mask, colour, label in [(is_g, KNOWN, "Generating class — capacity registered"),
                            (~is_g, ABSENT, "Non-generating class — no capacity field")]:
    sub = b[mask]
    fig.add_trace(go.Bar(
        y=sub.index, x=sub.rows, orientation="h", marker_color=colour, name=label,
        text=[f"{v}" for v in sub.rows], textposition="outside",
        textfont=dict(color=INK_2, size=11.5), cliponaxis=False,
        hovertemplate="%{y}<br>%{x} rows<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Bar(
        y=sub.index, x=sub.total_mw, orientation="h", marker_color=colour, showlegend=False,
        text=[f"{v:,.0f}" if v > 0 else "" for v in sub.total_mw], textposition="outside",
        textfont=dict(color=INK_2, size=11.5), cliponaxis=False,
        hovertemplate="%{y}<br>%{x:,.1f} MW<extra></extra>"), row=1, col=2)

# The zero bars would otherwise read as "not drawn" rather than "nothing to draw".
for cls in b.index[~is_g]:
    fig.add_annotation(x=0, y=cls, text="  no capacity field", xref="x2", yref="y2",
                       showarrow=False, xanchor="left",
                       font=dict(size=11, color=ABSENT))

fig.update_layout(barcornerradius=4, bargap=0.34)
fig.update_yaxes(categoryorder="array", categoryarray=order[::-1], showgrid=False)
fig.update_xaxes(range=[0, 104], row=1, col=1)
fig.update_xaxes(range=[0, 8600], row=1, col=2)
style(fig, "The largest class in the registry has no capacity",
      "176 rows by facility class. Demand Side Programme is 90 of them and 0 MW of the 8,805 MW "
      "registered; Scheduled Facility is 47 rows and 83% of the capacity.",
      height=470, left=170, top=152, legend_y=1.12)
# Drop the subplot titles clear of the legend row above them.
for a in fig.layout.annotations[:2]:
    a.update(y=1.045, font=dict(size=12.5, color=INK_2))
fig.show()

### 2. Capacity by technology, on a log scale

One mark per generating facility, one row per inferred technology, x on a **log** axis because
registered capacity spans from `BREMER_BAY_WF1` at 0.6 MW to `COLLIE_BESS2` at 600 MW — three orders
of magnitude, which a linear axis would collapse into a stack at the left edge and four dots at the
right. Rows are ordered by total capacity and the tick labels carry the facility count and total, so
the row that dominates the megawatts and the row that dominates the count can be told apart.

Technology is carried by **row position**, so colour is free for the thing that matters here: the
orange row is the eleven facilities whose technology the code convention cannot resolve. After the
three suffix fixes they are all small — the largest is 82 MW — so the unresolved share is 2.4% of
capacity rather than the 10.4% the shipped patterns left.

The shape of each row is the real content. **Thermal is sharply bimodal, with an empty band in the
middle**: 16 units at 43.4 MW or below, 20 units at 103.9 MW or above, and *not one facility between
those two figures*. The lower group is industrial cogeneration and small peaking plant, the upper is
the utility fleet, and nothing in the registry sits between the two. **Wind is small and long-tailed**
— 17 facilities, a median of 21.6 MW, and only four above 100 MW. **Storage is the opposite of
both**: seven facilities, none of them small, and its median of 450 MW would still be the highest in
the chart at 225 MW after the two-sided-span correction from section 4. That correction is
annotated on the row rather than applied to it, because this chart shows the registry as written.

In [9]:
# ── Chart 2: every generator, by inferred technology ─────────────────────────
tech_tot = (gen.groupby('tech2').system_size_mw.agg(n='count', mw='sum')
            .sort_values('mw'))          # ascending: plotly draws categories bottom-up
rows = list(tech_tot.index)
tick = {t: f"{t}<br><span style='font-size:10.5px'>{int(r.n)} facilities · "
           f"{r.mw:,.0f} MW</span>" for t, r in tech_tot.iterrows()}

# The y axis is NUMERIC, not categorical, because the marks are jittered off their
# row centre to stop the 36 thermal facilities overplotting each other.
ypos = {t: i for i, t in enumerate(rows)}
rng = np.random.default_rng(0)           # seeded: the jitter must not move between runs
fig = go.Figure()
for t in rows:
    sub = gen[gen.tech2 == t]
    colour = ABSENT if t == 'unknown' else KNOWN
    fig.add_trace(go.Scatter(
        x=sub.system_size_mw, y=ypos[t] + rng.uniform(-0.22, 0.22, len(sub)),
        mode="markers", name=t, showlegend=False,
        marker=dict(size=10, color=colour, opacity=0.72,
                    line=dict(width=1.5, color=SURFACE)),
        customdata=np.stack([sub.facility_code, sub.participant_name.map(shorten)], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>%{x:,.1f} MW<extra></extra>"))
    # Median tick: a thin vertical rule, the one summary statistic this form can
    # carry. Drawn as a trace rather than a shape because shape coordinates on a
    # log axis are read as raw data values while annotation coordinates are read
    # as log10 — a trace is unambiguous.
    med = sub.system_size_mw.median()
    fig.add_trace(go.Scatter(
        x=[med, med], y=[ypos[t] - 0.34, ypos[t] + 0.34], mode="lines", showlegend=False,
        line=dict(color=INK_2, width=2),
        hovertemplate=f"{t} median {med:,.1f} MW<extra></extra>"))

# Legend by proxy: two entries, because the orange row means something.
for colour, label in [(KNOWN, "technology resolved from the facility code"),
                      (ABSENT, "unresolved — code convention gives no clue")]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=label,
                             marker=dict(size=10, color=colour,
                                         line=dict(width=1.5, color=SURFACE))))

# Direct labels: the extremes of the range, and the correction this chart does not apply.
for code, dy in [("COLLIE_BESS2", -30), ("BREMER_BAY_WF1", 30),
                 ("NEWGEN_KWINANA_CCG1", 30)]:
    r_ = gen[gen.facility_code == code].iloc[0]
    fig.add_annotation(x=np.log10(r_.system_size_mw), y=ypos[r_.tech2], text=code,
                       showarrow=False, yshift=dy, font=dict(size=10.5, color=INK_2))
fig.add_annotation(x=np.log10(2.4), y=ypos['storage'], xanchor="left", showarrow=False,
                   text="registered two-sided — halve for discharge power (§4)",
                   font=dict(size=10.5, color=ABSENT))

fig.update_xaxes(type="log", title_text="registered System Size (MW), log scale",
                 range=[np.log10(0.38), np.log10(1400)],
                 tickvals=[0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500],
                 ticktext=["0.5", "1", "2", "5", "10", "20", "50", "100", "200", "500"])
fig.update_yaxes(showgrid=False, tickvals=list(ypos.values()),
                 ticktext=[tick[t] for t in rows], range=[-0.62, len(rows) - 0.38])
style(fig, "Three orders of magnitude, and eleven facilities the naming convention cannot place",
      "One mark per generating facility; the vertical rule on each row is that technology's median. "
      "Storage is shown as registered — see section 4.",
      height=480, left=150, legend_y=1.03)
fig.show()

### 3. Who owns it

Left, the concentration curve: participants ranked from largest to smallest, against the cumulative
share of registered capacity they account for. The diagonal is what perfect equality would look like
across the same 38 participants. Right, the ten largest holders, with the number of generating
facilities each one holds printed against the bar.

The curve is steep. **Synergy alone holds 44.5% of registered capacity across 27 generating
facilities, two participants hold half, and six hold 80%.** The Herfindahl–Hirschman index on
capacity share is **2,366** — just under the 2,500 that conventionally marks a highly concentrated
market, and that is before noting that the second-placed holder is a battery trust whose 1,000 MW is
the two-sided figure from section 4 and is really 500 MW of discharge power.

The right panel also shows why the facility count is the wrong denominator for almost every question
about this file. Seven of the top ten hold exactly **one** facility each. Enel X, which holds 87
facilities — more than three times as many as Synergy — does not appear on this panel at all, because
demand-response registrations carry no capacity. Count rows and Enel X is the dominant participant in
the WEM; count megawatts and it is not on the chart.

In [10]:
# ── Chart 3: concentration of registered capacity ────────────────────────────
p = (gen.groupby('participant_name').system_size_mw.agg(n='count', mw='sum')
     .sort_values('mw', ascending=False))
p['share'] = p.mw / TOTAL_MW
p['cum'] = p.share.cumsum()
HHI = (p.share ** 2).sum() * 1e4
n_50 = int((p.cum < 0.50).sum() + 1)
n_80 = int((p.cum < 0.80).sum() + 1)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13, column_widths=[0.44, 0.56],
                    subplot_titles=["Cumulative share of registered capacity",
                                    "Ten largest holders, MW (facility count at the bar)"])

x = np.arange(0, len(p) + 1) / len(p) * 100
y = np.concatenate([[0], p.cum.values * 100])
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", showlegend=False,
                         line=dict(color=AXIS, width=1.5, dash="dot"),
                         hoverinfo="skip"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="actual", showlegend=False,
                         line=dict(color=KNOWN, width=2, shape="hv"),
                         hovertemplate="top %{x:.0f}% of participants<br>"
                                       "%{y:.1f}% of capacity<extra></extra>"), row=1, col=1)
for k, lab in [(1, f"Synergy alone: {100 * p.cum.iloc[0]:.1f}%"),
               (n_50, f"{n_50} participants: 50%"),
               (n_80, f"{n_80} participants: 80%")]:
    fig.add_trace(go.Scatter(x=[k / len(p) * 100], y=[p.cum.iloc[k - 1] * 100], mode="markers",
                             marker=dict(size=9, color=KNOWN, line=dict(width=1.5, color=SURFACE)),
                             showlegend=False, hoverinfo="skip"), row=1, col=1)
    fig.add_annotation(x=k / len(p) * 100, y=p.cum.iloc[k - 1] * 100, text="  " + lab,
                       xref="x", yref="y", showarrow=False, xanchor="left", yanchor="top",
                       font=dict(size=10.5, color=INK_2))
fig.add_annotation(x=97, y=6, text=f"perfect equality<br>would be the dotted line<br><br>"
                                   f"<b>HHI {HHI:,.0f}</b>",
                   xref="x", yref="y", showarrow=False, xanchor="right", yanchor="bottom",
                   align="right", font=dict(size=11, color=MUTED))

top = p.head(10).iloc[::-1]
fig.add_trace(go.Bar(y=[shorten(i) for i in top.index], x=top.mw, orientation="h",
                     marker_color=KNOWN, showlegend=False,
                     text=[f"{v:,.0f} MW  ·  {int(n_)} " + ("facility" if n_ == 1 else "facilities")
                           for v, n_ in zip(top.mw, top.n)],
                     textposition="outside", textfont=dict(color=INK_2, size=11),
                     cliponaxis=False,
                     hovertemplate="%{y}<br>%{x:,.1f} MW<extra></extra>"), row=1, col=2)

fig.update_layout(barcornerradius=4, bargap=0.34)
fig.update_xaxes(range=[0, 100], ticksuffix="%", title_text="participants, ranked",
                 row=1, col=1)
fig.update_yaxes(range=[0, 104], ticksuffix="%", row=1, col=1)
fig.update_xaxes(range=[0, 5900], row=1, col=2)
fig.update_yaxes(showgrid=False, row=1, col=2)
style(fig, f"Two participants hold half the registered capacity",
      f"{len(p)} participants hold capacity. Enel X holds 87 facilities and none of it, so it is "
      f"absent from both panels.", height=470, left=64, showlegend=False)
fig.show()

### 4. What the registry can actually be joined to

Only two files in this project are per-facility, so only two can be joined to the registry at all:
the daily maximum temperature panel and the storage SCADA extract. This chart asks what fraction of
the registry each one reaches, measured two ways — by facility count, and by the capacity those
facilities represent — because the two answers are not close and the wrong one is discouraging.

**Facility temperature covers 58 of the 76 generators, 76.3% by count, but 98.9% by capacity.** The
18 generators with no temperature series are all small — the largest is `ALBANY_WF1` at 21.6 MW, and
two of the eighteen are the wind farms from section 2 that appear in the panel with no readings at
all. For any capacity-weighted question the temperature panel is effectively complete; for a
per-facility question about small embedded generation it is not.

The storage extract runs the other way: **7 facilities, 9.2% by count, 32.4% by capacity** — and
because it is a deliberate `_BESS`/`_ESR` filter rather than a gap, that is full coverage of what it
was built to cover. Every code in it resolves in the registry, and every storage facility in the
registry appears in it, which is the check worth having.

In [11]:
# ── Chart 4: join coverage, by count and by capacity ─────────────────────────
scada_codes = set(scada.facility_code.unique())
covered = {'Facility temperature': gen.facility_code.isin(live_codes),
           'Storage SCADA extract': gen.facility_code.isin(scada_codes)}

bars = []
for name, m in covered.items():
    bars.append((f"{name}<br><span style='font-size:10.5px'>by capacity</span>",
                 100 * gen.loc[m, 'system_size_mw'].sum() / TOTAL_MW,
                 f"{gen.loc[m, 'system_size_mw'].sum():,.0f} of {TOTAL_MW:,.0f} MW"))
    bars.append((f"{name}<br><span style='font-size:10.5px'>by facility count</span>",
                 100 * m.sum() / len(gen), f"{m.sum()} of {len(gen)} generators"))
labels = [b[0] for b in bars][::-1]
pct = [b[1] for b in bars][::-1]
note = [b[2] for b in bars][::-1]

fig = go.Figure()
fig.add_trace(go.Bar(y=labels, x=pct, orientation="h", name="in both files",
                     marker=dict(color=KNOWN, line=dict(color=SURFACE, width=2)),
                     text=[f"{v:.1f}%" for v in pct], textposition="inside",
                     insidetextanchor="end", textfont=dict(color=SURFACE, size=12),
                     customdata=note,
                     hovertemplate="%{customdata}<br>%{x:.1f}%<extra></extra>"))
fig.add_trace(go.Bar(y=labels, x=[100 - v for v in pct], orientation="h",
                     name="in the registry only",
                     marker=dict(color=ABSENT, line=dict(color=SURFACE, width=2)),
                     hovertemplate="%{x:.1f}% not covered<extra></extra>"))

fig.update_layout(barmode="stack", barcornerradius=4, bargap=0.42,
                  legend_traceorder="normal")
fig.update_xaxes(range=[0, 100], ticksuffix="%",
                 title_text="share of the registered generator fleet, on the basis named at the row")
fig.update_yaxes(showgrid=False)
style(fig, "Coverage by count and coverage by capacity are different questions",
      "The temperature panel misses a quarter of the generators and 1.1% of the megawatts. "
      "The storage extract is a deliberate filter, not a gap.",
      height=400, left=210, legend_y=1.04)
fig.show()

print('generators with no temperature series, largest first:')
print(gen[~gen.facility_code.isin(live_codes)]
      .nlargest(6, 'system_size_mw')[['facility_code', 'tech2', 'system_size_mw']]
      .to_string(index=False))
print(f"\nstorage in the registry {int((gen.tech2 == 'storage').sum())}   "
      f"in the SCADA extract {len(scada_codes)}   "
      f"resolving both ways {len(scada_codes & set(gen.loc[gen.tech2 == 'storage', 'facility_code']))}")

generators with no temperature series, largest first:
         facility_code   tech2  system_size_mw
            ALBANY_WF1    wind          21.600
          GRASMERE_WF1    wind          13.800
        NORTHAM_SF_PV1   solar           9.800
  BLAIRFOX_BEROSRD_WF1    wind           9.252
BLAIRFOX_WESTHILLS_WF3    wind           8.850
           TAMALA_PARK unknown           5.550

storage in the registry 7   in the SCADA extract 7   resolving both ways 7


## Limits

What this file does **not** contain, listed because each one is a question someone will try to answer
from it:

1. **No commissioning or retirement date.** The build-out of the battery fleet in the storage
   notebook had to be measured from first dispatch, not read from here.
2. **No energy rating.** A battery's MWh, the number that decides whether it can cover an evening
   peak, is absent; the storage notebook reconstructs it by integrating dispatch.
3. **No fuel or prime mover**, hence section 3. Every technology label in this project is inferred
   from a naming convention and is a stated assumption, not a fact from the source.
4. **No location, connection point or region.** The temperature panel is the only geographic
   information available, and it is indirect.
5. **No history.** One snapshot, no effective dates, and section 2 shows it already disagrees with a
   series that starts in 2024.

And two cautions that apply to numbers computed above rather than to absent columns:

6. **`System Size (MW)` is not one measurement.** Section 4 — two-sided for storage, one-sided for
   everything else. Any mix, share or capacity-factor denominator has to correct for it.
7. **Row counts are registrations, not plant.** 87 of 176 rows are one aggregator's demand-response
   registrations. Nothing that counts rows without filtering by class means what it appears to mean.

In [12]:
# The limits, made checkable rather than asserted.
print('columns present:', f.columns.tolist())
for want in ['commission', 'retire', 'date', 'fuel', 'technology', 'mwh', 'energy',
             'lat', 'lon', 'region', 'zone', 'connection']:
    hit = [c for c in load_facilities(raw='../data/raw').columns if want in c.lower()]
    print(f'  {want:<12} -> {hit if hit else "absent"}')

print(f'\n7. row counts are registrations: {len(f)} rows, '
      f'{(f.facility_class == "Demand Side Programme").sum()} of them Demand Side Programme, '
      f'held by {f.loc[f.facility_class == "Demand Side Programme", "participant_name"].nunique()} participants')
print(f.loc[f.facility_class == 'Demand Side Programme', 'participant_name']
      .value_counts().to_string())

columns present: ['participant_code', 'participant_name', 'facility_code', 'facility_class', 'system_size_mw', 'remaining_mw', 'tech', 'is_gen', 'tech2', 'suffix']
  commission   -> absent
  retire       -> absent
  date         -> absent
  fuel         -> absent
  technology   -> absent
  mwh          -> absent
  energy       -> absent
  lat          -> absent
  lon          -> absent
  region       -> absent
  zone         -> absent
  connection   -> absent

7. row counts are registrations: 176 rows, 90 of them Demand Side Programme, held by 4 participants
participant_name
Enel X Australia Pty Ltd            87
Bluewaters Power 1 Pty Ltd           1
Synergy                              1
Wesfarmers Kleenheat Gas Pty Ltd     1


## What this notebook establishes

- **The registry is a clean dimension table with sound keys.** 176 rows, 176 unique facility codes,
  no duplicates, participant code and name in strict 1:1 correspondence. Every join in the series
  rests on this and it holds.
- **Its 100 null capacities are a definition, not a gap.** `System Size (MW)` is populated on exactly
  the 76 generating rows and nowhere else, and `remaining_mw` is populated on 5 of the 7
  Non-Dispatchable Loads and nowhere else. Filter by class; never impute.
- **It is a current snapshot and it will silently drop retired plant.** `MUJA_G6` has 454 temperature
  readings ending 31 March 2025 and no registry row. Use a left join and check the anti-join.
- **The inferred technology column had a 917 MW hole, now 211 MW.** Adding `_CCG`, `_COG` and `_WWF`
  to `TECH_SUFFIX` takes the unresolved share of capacity from 10.4% to 2.4% and stops the two
  largest combined-cycle gas plants in the state from being classed as `unknown`. Eleven small
  facilities remain genuinely uninferable and are left that way.
- **Storage capacity is stated on a two-sided basis and is not comparable to the rest of the
  column.** Measured against dispatch, `System Size` for six of seven batteries is charge plus
  discharge power to within 1%. On a discharge basis the fleet is 1,425 MW, not 2,850, and the
  registered mix of 48.7% thermal / 32.4% storage becomes 58.1% / 19.3%.
- **Concentration is high and the row count conceals it.** 38 participants hold the 8,805 MW; Synergy
  holds 44.5%, two hold half, six hold 80%, HHI 2,366. The participant with the most rows in the file
  holds none of the capacity.

What this hands the rest of the series: a `tech2` column that resolves 97.6% of registered capacity,
a documented correction for the storage basis, and the knowledge that a capacity-weighted question
about the temperature panel is answerable to 98.9% coverage while a facility-weighted one is not.